In [2]:
import os
from PIL import Image
import torch
from diffusers import StableDiffusionInstructPix2PixPipeline, EulerAncestralDiscreteScheduler

/home/allwardt/smart_graphics_submission_2/torch_venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
INPUT_IMG_PATH = './images/input/wwz.jpeg'
OUTPUT_DIR_PATH = './images/output/'
MODEL_ID = "timbrooks/instruct-pix2pix"

In [4]:
pipe = StableDiffusionInstructPix2PixPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.float16, safety_checker=None)
pipe.to("cuda")
pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)

Loading pipeline components...: 100%|██████████| 6/6 [00:01<00:00,  3.67it/s]


RuntimeError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
prompt = "remove the trees"

In [1]:
def resize_image_keep_aspect(image, target_size=256):
    w, h = image.size
    if w > h:
        # Querformat: Breite wird 256, Höhe wird proportional skaliert
        new_w = target_size
        new_h = int(h * (target_size / w))
    else:
        # Hochformat oder quadratisch: Höhe wird 256, Breite wird skaliert
        new_h = target_size
        new_w = int(w * (target_size / h))
    return image.resize((new_w, new_h), Image.ANTIALIAS)

In [ ]:
image = Image.open(INPUT_IMG_PATH).convert("RGB")
resized = resize_image_keep_aspect(image)
images = pipe(prompt, image=image, num_inference_steps=10, image_guidance_scale=1).images
images[0]

100%|██████████| 10/10 [00:09<00:00,  1.02it/s]


In [26]:
with open(os.path.join(OUTPUT_DIR_PATH, 'roland_snowman.png'), 'wb') as f:
    images[0].save(f, format='PNG')